In [1]:
# df.columns = df.columns.str.strip()
# print(df.columns.tolist())

In [2]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split

In [3]:
from pathlib import Path
data_path = Path("Model V1") / "urldata.csv"
if not data_path.exists():
    data_path = Path("urldata.csv")
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print(df.head())

Columns: ['Unnamed: 0', 'url', 'label', 'result']
Shape: (450180, 4)
   Unnamed: 0                        url   label  result
0           0     https://www.google.com  benign       0
1           1    https://www.youtube.com  benign       0
2           2   https://www.facebook.com  benign       0
3           3      https://www.baidu.com  benign       0
4           4  https://www.wikipedia.org  benign       0


In [4]:
df = df[['url', 'result']].dropna()
df.columns = ['url', 'label']
df['label'] = df['label'].astype(int)
print(f"Total    : {len(df)}")
print(f"Benign   : {(df['label']==0).sum()}")
print(f"Malicious: {(df['label']==1).sum()}")
print(df.head())

Total    : 450180
Benign   : 345742
Malicious: 104438
                         url  label
0     https://www.google.com      0
1    https://www.youtube.com      0
2   https://www.facebook.com      0
3      https://www.baidu.com      0
4  https://www.wikipedia.org      0


In [5]:
def extract_features(url):
    url = str(url).lower().strip()
    features = [
        len(url),
        url.count('.'),
        url.count('-'),
        url.count('/'),
        sum(c.isdigit() for c in url),
        len(re.findall(r'[^a-z0-9.\-/:]', url)),
        1 if url.startswith('https') else 0,
        len(url.split('/')[2]) if '//' in url else len(url),
        1 if any(t in url for t in ['.tk','.xyz','.ru','.pw']) else 0,
        len(url.split('/')[-1]),
    ]
    return features

print("Extracting features...")
X = np.array([extract_features(u) for u in df['url']], dtype=np.float32)
y = df['label'].values.astype(np.float32)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Extracting features...
X shape: (450180, 10)
y shape: (450180,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (360144, 10)  |  Test: (90036, 10)


In [7]:
model = Sequential([
    Input(shape=(10,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1,  activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9734 - loss: 0.0975 - val_accuracy: 0.9858 - val_loss: 0.0654
Epoch 2/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9853 - loss: 0.0620 - val_accuracy: 0.9868 - val_loss: 0.0547
Epoch 3/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.9859 - loss: 0.0560 - val_accuracy: 0.9872 - val_loss: 0.0492
Epoch 4/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9873 - loss: 0.0500 - val_accuracy: 0.9880 - val_loss: 0.0482
Epoch 5/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.9879 - loss: 0.0480 - val_accuracy: 0.9880 - val_loss: 0.0481
Epoch 6/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.9884 - loss: 0.0461 - val_accuracy: 0.9886 - val_loss: 0.0451
Epoch 7/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.9885 - loss: 0.0451 - val_accuracy: 0.9881 - val_loss: 0.0489
Epoch 8/10
5628/5628 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.9887 - loss: 0

In [8]:
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")

2814/2814 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9902 - loss: 0.0408

✅ Test Accuracy: 99.02%


In [9]:
from pathlib import Path
model_path = Path("Model V1") / "url_spam_detector.h5"
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save(model_path)
print("✅ Saved as url_spam_detector.h5")

✅ Saved as url_spam_detector.h5


In [12]:
test_urls = [
    "https://www.google.com",
    "http://free-money.tk/click/win",
    "https://www.amazon.com/products/item",
    "http://192.168.1.1/phishing/login.php",
    "www.instagram.com"
]

print("\n--- Manual Test Results ---")
for url in test_urls:
    features   = np.array([extract_features(url)], dtype=np.float32)
    prediction = model.predict(features, verbose=0)[0][0]
    is_spam    = prediction >= 0.5
    label      = "🚨 SPAM/PHISHING" if is_spam else "✅ SAFE"
    confidence = prediction if is_spam else 1 - prediction
    print(f"{label} ({confidence*100:.1f}%) — {url}")


--- Manual Test Results ---
✅ SAFE (99.5%) — https://www.google.com
🚨 SPAM/PHISHING (100.0%) — http://free-money.tk/click/win
✅ SAFE (98.0%) — https://www.amazon.com/products/item
🚨 SPAM/PHISHING (99.9%) — http://192.168.1.1/phishing/login.php
🚨 SPAM/PHISHING (98.5%) — www.instagram.com
